# Exploratory Data Analysis: The Phillips Curve Across U.S. States

## Texas, Massachusetts, and Ohio — 2000 to Present

This notebook explores the data collected in notebook `01_data_collection.ipynb` and builds visualizations of the unemployment–inflation relationship across three state economies before any formal regression analysis is run in Week 3.

The goal here is to develop visual and statistical intuition about how the Phillips Curve behaves at the state level — to *see* the relationship in scatter plots, time series, and summary statistics before we attempt to estimate it. By the end of this notebook we will have a set of presentation-quality figures and a stronger sense of where the data agrees with textbook theory and where it surprises us.

In [ ]:
import os
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

sns.set_context('notebook', font_scale=1.2)

print('Libraries loaded successfully.')
print(f'  pandas      {pd.__version__}')
print(f'  numpy       {np.__version__}')
print(f'  seaborn     {sns.__version__}')

In [ ]:
panel = pd.read_csv('../data/processed/phillips_curve_panel.csv')
state_chars = pd.read_csv('../data/processed/state_characteristics.csv')

panel['date'] = pd.to_datetime(panel['date'])

print(f'Panel shape: {panel.shape[0]:,} rows x {panel.shape[1]} columns')
print(f'\nColumns:')
for col in panel.columns:
    print(f'  - {col}')

print(f'\nDate range: {panel["date"].min().date()} to {panel["date"].max().date()}')

print(f'\nObservations per state:')
print(panel.groupby('state').size().to_string())

print(f'\nState characteristics:')
print(state_chars.to_string(index=False))

print(f'\nFirst 5 rows of panel:')
panel.head()

## Summary Statistics by State

Before plotting, we want a quantitative snapshot of how unemployment and inflation behave in each state. These tables describe the *marginal* distributions — the level and spread of each variable on its own — and provide the baseline against which the joint Phillips Curve relationship will be judged.

In [ ]:
summary_by_state = (
    panel
    .groupby('state')[['unemployment_rate', 'inflation_rate_yoy']]
    .describe()
    .round(2)
)

summary_by_state

### Key differences across the three states

**Ohio has run the hottest labor market on average** — its mean unemployment rate of 5.89% over 2000–present sits roughly three quarters of a point above Texas (5.40%) and Massachusetts (5.15%), consistent with its slower-growing, manufacturing-heavy economy. Texas has the *tightest* labor market when measured by spread: a standard deviation of just 1.52, versus ~1.9 for both MA and OH, reflecting Texas's relatively continuous job growth even through national downturns.

**Inflation volatility is highest in Texas and Ohio** (standard deviations of 1.95 each), while Massachusetts is noticeably more stable at 1.66. That gap is driven by the extremes: Texas hit a YoY inflation peak above 10% during the 2021–22 spike, while MA's peak stayed around 8%. On the downside, all three states briefly touched deflation around 2009 and again in 2020.

**The unemployment range is dominated by the COVID shock.** Each state's maximum unemployment (MA 17.8%, OH 16.5%, TX 12.8%) is a pandemic outlier — Texas's much lower peak previews one of the cleaner narratives in this data: TX's labor market absorbed COVID with substantially less disruption than the older industrial economies of MA and OH.

In [ ]:
def assign_era(d):
    if d < pd.Timestamp('2008-01-01'):
        return 'Pre-Crisis (2000-07)'
    if d < pd.Timestamp('2010-01-01'):
        return 'Great Recession (2008-09)'
    if d < pd.Timestamp('2020-01-01'):
        return 'Long Expansion (2010-19)'
    return 'COVID & Aftermath (2020+)'

panel['era'] = panel['date'].apply(assign_era)

era_order = [
    'Pre-Crisis (2000-07)',
    'Great Recession (2008-09)',
    'Long Expansion (2010-19)',
    'COVID & Aftermath (2020+)',
]

summary_by_era = (
    panel
    .groupby(['era', 'state'])[['unemployment_rate', 'inflation_rate_yoy']]
    .mean()
    .unstack('state')
    .round(2)
    .reindex(era_order)
)

summary_by_era.columns = [
    f'{"unemp" if "unemployment" in var else "infl"} — {state}'
    for var, state in summary_by_era.columns
]

summary_by_era

## Time Series: Unemployment and Inflation by State

Before we look at the joint scatter of unemployment vs. inflation, it helps to see each series in isolation, on its own timeline. The Phillips Curve predicts that the two variables move in *opposite* directions: when unemployment is low, inflation should be high, and vice versa. By plotting both series for each state, we can scan the historical record for periods where they did move opposite each other (consistent with the theory) and periods where they moved together (inconsistent — typically a supply-side shock).

NBER recession periods are shaded in light gray, so we can quickly anchor the swings to the macroeconomic backdrop.

In [ ]:
UNEMP_COLOR = '#1f3a93'   # dark blue
INFL_COLOR = '#a52a2a'    # dark red

state_order = [
    ('TX', 'Texas'),
    ('MA', 'Massachusetts'),
    ('OH', 'Ohio'),
]

recessions = [
    ('2001-03-01', '2001-11-30'),
    ('2007-12-01', '2009-06-30'),
    ('2020-02-01', '2020-04-30'),
]

fig, axes = plt.subplots(3, 1, figsize=(14, 14), sharex=True)

for ax_u, (code, name) in zip(axes, state_order):
    df = panel[panel['state'] == code].sort_values('date')

    ax_u.plot(df['date'], df['unemployment_rate'],
              color=UNEMP_COLOR, linewidth=2, label='Unemployment Rate')
    ax_u.set_ylabel('Unemployment Rate (%)', color=UNEMP_COLOR)
    ax_u.tick_params(axis='y', labelcolor=UNEMP_COLOR)
    ax_u.tick_params(axis='x', labelbottom=True)
    ax_u.set_xlabel('Date')
    ax_u.set_title(name, fontsize=14, fontweight='bold')

    ax_i = ax_u.twinx()
    ax_i.plot(df['date'], df['inflation_rate_yoy'],
              color=INFL_COLOR, linewidth=2, label='Inflation Rate (YoY)')
    ax_i.set_ylabel('Inflation Rate YoY (%)', color=INFL_COLOR)
    ax_i.tick_params(axis='y', labelcolor=INFL_COLOR)
    ax_i.axhline(0, color=INFL_COLOR, linestyle='--', linewidth=1, alpha=0.6)
    ax_i.grid(False)

    for start, end in recessions:
        ax_u.axvspan(pd.Timestamp(start), pd.Timestamp(end),
                     color='gray', alpha=0.2, zorder=0)

    ax_u.xaxis.set_major_locator(mdates.YearLocator(2))
    ax_u.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

handles = [
    plt.Line2D([0], [0], color=UNEMP_COLOR, linewidth=2, label='Unemployment Rate'),
    plt.Line2D([0], [0], color=INFL_COLOR, linewidth=2, label='Inflation Rate (YoY)'),
    plt.Rectangle((0, 0), 1, 1, facecolor='gray', alpha=0.2, label='NBER Recession'),
]
fig.legend(handles=handles, loc='upper center', ncol=3,
           bbox_to_anchor=(0.5, 0.995), frameon=False, fontsize=12)

fig.suptitle('Unemployment and Inflation Over Time', fontsize=16, fontweight='bold', y=1.02)
fig.tight_layout(rect=[0, 0, 1, 0.97])

os.makedirs('../figures', exist_ok=True)
fig.savefig('../figures/time_series_unemployment_inflation.png',
            dpi=200, bbox_inches='tight')

plt.show()

### What the time series shows

**The 2008–2009 recession is the cleanest Phillips Curve episode in the sample.** In all three states, unemployment spikes sharply (especially Ohio, which crosses 10%) while year-over-year inflation collapses — briefly turning negative in 2009. This is exactly what a textbook negative demand shock looks like: the labor market loosens and price pressure evaporates simultaneously.

**The 2021–2022 episode tells the opposite story.** Inflation surges to 8–10% YoY across all three states while unemployment is *falling* rapidly out of the COVID trough. A naive Phillips Curve interpretation would have called this combination impossible — but it is the canonical signature of a supply shock layered on top of a recovering labor market, and it is the empirical puzzle that motivates almost every modern reassessment of the curve.

**Texas shows visibly more inflation amplitude than Massachusetts.** The Texas inflation series swings noticeably wider during oil-driven episodes (2008, 2014–15, and especially 2021–22), reflecting energy's larger share of the state's price basket and economy. Massachusetts, with a service- and education-heavy economy, has a smoother inflation series.

**Ohio's unemployment cycle is the most pronounced.** Both the 2009 peak and the COVID peak are larger in Ohio than in Texas — consistent with Ohio's greater exposure to durable-goods manufacturing, a sector that contracts most violently in recessions. Texas, by contrast, exhibits the shallowest unemployment cycle, hinting that the *slope* of any estimated Phillips Curve may differ meaningfully across these three economies.

## Control Variables Over Time

The Phillips Curve relationship can be distorted by national and global forces that move inflation independently of any given state's labor market. Before we look at the joint scatter, we plot the four main control variables — oil prices, mortgage rates, the federal funds rate, and national CPI inflation — so we can see when those shocks were active.

In [ ]:
CONTROL_COLOR = '#0b1f4d'  # dark navy

national = (
    panel[['date', 'oil_price_wti', 'mortgage_rate_30yr',
           'fed_funds_rate', 'national_inflation_yoy']]
    .drop_duplicates(subset='date')
    .sort_values('date')
)

controls = [
    ('oil_price_wti',         'WTI Crude Oil Price ($/barrel)',  'USD per barrel'),
    ('mortgage_rate_30yr',    '30-Year Fixed Mortgage Rate (%)', 'Percent'),
    ('fed_funds_rate',        'Federal Funds Rate (%)',          'Percent'),
    ('national_inflation_yoy','National CPI Inflation YoY (%)',  'Percent'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (col, title, ylabel) in zip(axes.flat, controls):
    ax.plot(national['date'], national[col],
            color=CONTROL_COLOR, linewidth=2)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel(ylabel)

    for start, end in recessions:
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end),
                   color='gray', alpha=0.2, zorder=0)

    if col == 'national_inflation_yoy':
        ax.axhline(0, color=CONTROL_COLOR, linestyle='--', linewidth=1, alpha=0.5)

    ax.xaxis.set_major_locator(mdates.YearLocator(4))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

fig.suptitle('National & Global Control Variables (2000–Present)',
             fontsize=16, fontweight='bold', y=1.00)
fig.tight_layout()

fig.savefig('../figures/control_variables_time_series.png',
            dpi=200, bbox_inches='tight')

plt.show()

### Why these variables matter as controls

The Phillips Curve is, in its simplest form, a claim that *local* labor-market slack drives *local* price changes. But state-level inflation is also pushed around by national and global forces that have nothing to do with the unemployment rate in Texas, Massachusetts, or Ohio. If we ignore them, we risk attributing those movements to the labor market when they actually reflect something else:

- **WTI crude oil price** is the cleanest example of a supply shock. When oil doubles, gasoline and transport costs feed into every state's CPI within months — independent of how tight the local labor market is. The 2008 spike, the 2014–15 collapse, the 2020 crash, and the 2022 surge are all visible.
- **30-year mortgage rate** captures financing conditions that affect housing costs and aggregate demand. Housing is one of the largest components of CPI, so mortgage rates indirectly shape measured inflation.
- **Federal funds rate** is the proximate instrument of monetary policy. It moves with the Fed's view of where inflation and unemployment are headed *nationally*, and it is the main lever closing or opening the gap between local labor conditions and realized inflation.
- **National CPI inflation YoY** is the most direct control for national price pressure. Including it lets us isolate the component of *state* inflation that is genuinely above or below the national trend — which is the part the local labor market can plausibly explain.

In the regression in Week 3, these four controls will let us ask the sharper question: holding national inflation, monetary policy, oil, and housing costs constant, does local unemployment still move local inflation in the direction the Phillips Curve predicts?

## The Phillips Curve: Unemployment vs. Inflation by State

This is the signature visualization of the project. The Phillips Curve, in its original 1958 form, was an empirical claim: there is an inverse relationship between unemployment (on the x-axis) and the rate of price change (on the y-axis). If the relationship still holds, the cloud of monthly observations should slope *downward* — periods of low unemployment should line up with periods of high inflation, and vice versa.

We fit a simple OLS regression line for each state separately. The slope coefficient tells us how much inflation moves, on average, for a one-percentage-point change in unemployment. The R² tells us how much of inflation's variation a single labor-market variable can explain on its own — a critical benchmark, because if R² is low it means we *need* the controls and the panel structure we'll layer in during Week 3.

In [ ]:
from scipy import stats

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

x_min = panel['unemployment_rate'].min() - 0.5
x_max = panel['unemployment_rate'].max() + 0.5
x_grid = np.linspace(x_min, x_max, 200)

for ax, (code, name) in zip(axes, state_order):
    df = (panel[panel['state'] == code]
          [['unemployment_rate', 'inflation_rate_yoy']]
          .dropna())

    ax.scatter(df['unemployment_rate'], df['inflation_rate_yoy'],
               s=20, alpha=0.4, color='#1f3a93', edgecolors='none')

    reg = stats.linregress(df['unemployment_rate'], df['inflation_rate_yoy'])
    slope, intercept = reg.slope, reg.intercept
    r2 = reg.rvalue ** 2
    pval = reg.pvalue

    y_hat = intercept + slope * x_grid
    n = len(df)
    x_mean = df['unemployment_rate'].mean()
    ss_x = ((df['unemployment_rate'] - x_mean) ** 2).sum()
    residuals = df['inflation_rate_yoy'] - (intercept + slope * df['unemployment_rate'])
    s_err = np.sqrt((residuals ** 2).sum() / (n - 2))
    se_yhat = s_err * np.sqrt(1.0 / n + (x_grid - x_mean) ** 2 / ss_x)
    t_crit = stats.t.ppf(0.975, df=n - 2)
    ci = t_crit * se_yhat

    ax.fill_between(x_grid, y_hat - ci, y_hat + ci,
                    color='#d62728', alpha=0.18, label='95% CI')
    ax.plot(x_grid, y_hat, color='#d62728', linewidth=2.2, label='OLS fit')

    ax.axhline(2, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.7)

    p_str = f'{pval:.2e}' if pval < 0.001 else f'{pval:.3f}'
    text = (f'$\\beta_1$ = {slope:+.3f}\n'
            f'$R^2$ = {r2:.3f}\n'
            f'$p$ = {p_str}\n'
            f'$n$ = {n}')
    ax.text(0.97, 0.97, text, transform=ax.transAxes,
            ha='right', va='top', fontsize=11,
            bbox=dict(boxstyle='round,pad=0.4',
                      facecolor='white', edgecolor='gray', alpha=0.9))

    ax.set_title(name, fontsize=14, fontweight='bold')
    ax.set_xlabel('Unemployment Rate (%)')
    ax.set_xlim(x_min, x_max)

axes[0].set_ylabel('YoY Inflation Rate (%)')

fig.suptitle('The Phillips Curve Across Three U.S. States (2000–Present)',
             fontsize=16, fontweight='bold', y=1.02)
fig.tight_layout()

fig.savefig('../figures/phillips_curve_basic.png',
            dpi=200, bbox_inches='tight')

plt.show()

### Interpretation

**Texas.** The pooled slope is **−0.354** with p ≈ 10⁻⁶ — statistically a textbook Phillips Curve relationship. But the R² is only **0.077**: less than 8% of the monthly variation in Texas inflation is explained by Texas unemployment alone. That is consistent with the time-series picture: Texas's CPI is dragged around by oil prices to a much larger extent than its labor market. The slope is correctly signed, but a single labor variable is plainly missing most of the action — exactly the case for the controls and the panel structure we will add next.

**Massachusetts.** The slope is essentially identical in magnitude (**−0.346**, p ≈ 10⁻¹²) but the fit is the **strongest of the three states**, with an R² of **0.153**. This makes sense: Massachusetts has a service-heavy, education-anchored economy where labor market tightness translates into wage and price pressure more cleanly, and the inflation series is less contaminated by energy noise. The Phillips Curve looks the most "alive" in MA in the raw data.

**Ohio.** Slope **−0.358**, R² **0.125**, p ≈ 10⁻¹⁰. Ohio sits between Texas and Massachusetts on fit quality. The negative slope is well-identified and the magnitude is similar across all three states — the Phillips Curve relationship is present in Ohio, but the manufacturing-heavy labor market also produces larger unemployment swings (visible as a wider horizontal spread in the scatter) that probably reflect demand shocks the simple curve can't disentangle from supply shocks.

**The cross-state takeaway.** All three slopes are negative, statistically significant, and remarkably similar in magnitude (≈ −0.35). On the raw scatter, the Phillips Curve is alive. But the low R² values (8%–15%) make the same point three different ways: unemployment alone explains only a small fraction of state-level inflation, and the bulk of the variation lives in oil shocks, monetary policy, national price pressure, and era-specific regime shifts. That is the empirical case for the multivariate panel regression in Week 3.

## The Phillips Curve by Economic Era

A single regression line through 25 years of monthly data masks a story economists have been arguing about for the last decade: the Phillips Curve appears to have **shifted, flattened, and re-steepened** across different macroeconomic regimes. By coloring each monthly observation by its era and fitting a separate line within each era, we can see whether the underlying relationship has moved over time — and whether that movement looks the same in Texas, Massachusetts, and Ohio.

In [ ]:
def assign_era_short(d):
    if d < pd.Timestamp('2008-01-01'):
        return 'Pre-Crisis'
    if d < pd.Timestamp('2010-01-01'):
        return 'Great Recession'
    if d < pd.Timestamp('2020-01-01'):
        return 'Long Expansion'
    return 'COVID & Aftermath'

if 'era_short' not in panel.columns:
    panel['era_short'] = panel['date'].apply(assign_era_short)

ERA_ORDER = ['Pre-Crisis', 'Great Recession', 'Long Expansion', 'COVID & Aftermath']
ERA_COLORS = {
    'Pre-Crisis':         '#1f77b4',  # blue
    'Great Recession':    '#d62728',  # red
    'Long Expansion':     '#2ca02c',  # green
    'COVID & Aftermath':  '#ff7f0e',  # orange
}

fig, axes = plt.subplots(1, 3, figsize=(18, 7), sharex=True, sharey=True)

x_min = panel['unemployment_rate'].min() - 0.5
x_max = panel['unemployment_rate'].max() + 0.5
y_min = panel['inflation_rate_yoy'].min() - 1
y_max = panel['inflation_rate_yoy'].max() + 1

era_slopes = {}

for ax, (code, name) in zip(axes, state_order):
    df_state = panel[panel['state'] == code]

    for era in ERA_ORDER:
        df_e = df_state[(df_state['era_short'] == era)
                        & df_state['inflation_rate_yoy'].notna()
                        & df_state['unemployment_rate'].notna()]
        if len(df_e) == 0:
            continue

        color = ERA_COLORS[era]
        ax.scatter(df_e['unemployment_rate'], df_e['inflation_rate_yoy'],
                   color=color, alpha=0.5, s=35, edgecolors='none', label=era)

        if len(df_e) >= 3:
            reg = stats.linregress(df_e['unemployment_rate'], df_e['inflation_rate_yoy'])
            x_line = np.linspace(df_e['unemployment_rate'].min(),
                                 df_e['unemployment_rate'].max(), 50)
            y_line = reg.intercept + reg.slope * x_line
            ax.plot(x_line, y_line, color=color, linewidth=2.5)
            era_slopes[(code, era)] = (reg.slope, reg.rvalue ** 2, reg.pvalue)

    ax.axhline(2, color='gray', linestyle='--', linewidth=1, alpha=0.6)
    ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.6)
    ax.set_title(name, fontsize=14, fontweight='bold')
    ax.set_xlabel('Unemployment Rate (%)')
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

axes[0].set_ylabel('YoY Inflation Rate (%)')

handles = [plt.Line2D([0], [0], marker='o', linestyle='-', linewidth=2.5,
                      color=ERA_COLORS[e], markersize=9, label=e)
           for e in ERA_ORDER]
fig.legend(handles=handles, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.02), frameon=False, fontsize=12)

fig.suptitle('How the Phillips Curve Has Shifted Over Time',
             fontsize=16, fontweight='bold', y=1.02)
fig.tight_layout(rect=[0, 0.04, 1, 0.98])

fig.savefig('../figures/phillips_curve_by_era.png',
            dpi=200, bbox_inches='tight')

plt.show()

print('\nEra-specific slopes (state, era): slope, R^2, p-value')
for (code, era), (slope, r2, pval) in era_slopes.items():
    print(f'  {code} | {era:<20} slope={slope:+.3f}  R^2={r2:.3f}  p={pval:.3g}')

### What the era-colored scatter reveals

This is, in some ways, the most consequential figure in the project. The pooled regression we just ran above is essentially an average over four very different macroeconomic regimes — and the era breakout shows that the *single-line* story conceals more than it reveals.

**1. The Great Recession is the textbook Phillips Curve.** The red 2008–09 dots trace a strikingly clean negative line in all three states, with slopes around **−1.2 to −1.3** and R² values of **0.57–0.69** — far steeper and far better-fit than anything else in the sample. When the economy is hit by a large negative demand shock, unemployment shoots up *and* inflation collapses, simultaneously, exactly as Phillips described. This is what the curve looks like when there is no supply shock muddying the picture.

**2. The Phillips Curve flattened — and in TX/OH actually inverted — during the Long Expansion (2010–2019).** The green lines tell a very different story. In Massachusetts, the slope is essentially **zero** (+0.017, p = 0.72): from 2010 to 2019, unemployment fell from ~8% to ~3% while MA inflation barely moved off 2%. In Texas (+0.249) and Ohio (+0.258), the slope is actually *positive* and significant — inflation drifted slightly *up* as unemployment fell, but the relationship is so weak that the dots form a cloud, not a line. **This is the empirical core of the "is the Phillips Curve dead?" debate of the late 2010s**, and the state-level data reproduces the national finding cleanly.

**3. COVID & Aftermath shows an upward *shift* of the curve, not just movement along it.** The orange dots sit at *similar unemployment levels* to the green Long-Expansion dots (3–6%) but at **dramatically higher inflation** (often 6–10% vs the ~2% norm). If the Phillips Curve were stable, inflation at these unemployment levels should have stayed near 2% — instead, the entire orange cluster lives well above the green cluster. This is the signature of a **supply-side shift**: the curve has moved up the page, driven by goods shortages, supply chains, and energy, not by a tighter labor market.

**4. Pre-Crisis shows a mild but recognizable negative slope** in all three states (−0.24 to −0.43), with R² between 0.02 and 0.05. The relationship was visible but weak — consistent with the picture that even before 2008, the Phillips Curve was holding only loosely once you condition on a single state.

**5. State-specific COVID patterns.** The orange cluster differs across states. Texas's COVID-era slope is the steepest negative of the three (**−0.72**), with the highest peak inflation — consistent with TX's outsized exposure to the 2021–22 oil price surge. Massachusetts's COVID slope is the shallowest (**−0.39**); its inflation rose, but less, and its labor market recovered along a flatter path. Ohio sits in between (**−0.58**). The state characteristics matter exactly where we predicted: TX's energy exposure amplified the supply shock, and MA's service-heavy economy buffered it.

**The takeaway for Week 3.** A single pooled regression is the wrong specification. The slope and intercept of the Phillips Curve clearly move over time and across states. The Week 3 panel regression will need state fixed effects (to absorb level differences across MA/OH/TX), era controls or time trends (to absorb the regime shifts visible here), and oil/national-inflation controls (to strip out the supply shocks that drove the COVID shift). Only then can we get a clean estimate of the *labor-market* component of the Phillips Curve.

## Correlation Analysis

Before running formal regressions, we want to see how *all* the variables in our panel correlate with one another within each state. Two questions matter here. First, which controls are most strongly associated with state-level inflation — those are the ones most worth including. Second, are any of the candidate controls dangerously correlated *with each other*? If two regressors carry essentially the same information (correlation above |0.7|), including both can cause **multicollinearity** — coefficients become noisy and hard to interpret, even though the joint fit looks fine. The heatmaps below let us spot both issues quickly.

In [ ]:
corr_vars = [
    'unemployment_rate',
    'inflation_rate_yoy',
    'oil_price_wti',
    'mortgage_rate_30yr',
    'fed_funds_rate',
    'national_inflation_yoy',
    'hpi',
]

pretty_labels = {
    'unemployment_rate':       'Unemp.',
    'inflation_rate_yoy':      'Inflation (YoY)',
    'oil_price_wti':           'Oil (WTI)',
    'mortgage_rate_30yr':      'Mortgage 30y',
    'fed_funds_rate':          'Fed Funds',
    'national_inflation_yoy':  'Natl. Inflation',
    'hpi':                     'HPI',
}

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

for ax, (code, name) in zip(axes, state_order):
    df = panel[panel['state'] == code][corr_vars].dropna()
    corr = df.corr()
    corr.index = [pretty_labels[c] for c in corr.index]
    corr.columns = [pretty_labels[c] for c in corr.columns]

    sns.heatmap(corr, ax=ax, annot=True, fmt='.2f',
                cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                square=True, cbar=True,
                annot_kws={'size': 10},
                linewidths=0.5, linecolor='white')
    ax.set_title(name, fontsize=14, fontweight='bold')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right')
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

fig.suptitle('Correlation Matrices by State',
             fontsize=16, fontweight='bold', y=1.02)
fig.tight_layout()

fig.savefig('../figures/correlation_heatmaps.png',
            dpi=200, bbox_inches='tight')

plt.show()

### Key findings from the correlation matrices

**Phillips Curve (unemployment ↔ inflation).** The unemployment–inflation correlation is negative in every state, consistent with the curve: **TX = −0.28, MA = −0.39, OH = −0.35**. Massachusetts again shows the cleanest single-variable relationship. None of these is anywhere near the magnitude of −1.0 — confirming once more that unemployment alone explains a modest fraction of state inflation.

**Oil prices and state inflation.** Oil correlates with state inflation at **TX = 0.43, OH = 0.43, MA = 0.23**. Texas and Ohio show essentially the same oil-sensitivity in their CPI baskets, while Massachusetts is noticeably less oil-driven — a clean reflection of MA's service-heavy economy versus the more goods/transport-heavy baskets in TX and OH. The asymmetry in *magnitude* between MA and TX/OH is exactly what we'd expect: oil shocks pass through to local prices more strongly where oil and gas are larger shares of consumption and production.

**Multicollinearity warnings.** Two correlations are large enough to flag for the Week 3 regression:

- **National inflation ↔ state inflation: ~0.92–0.97** in every state. This is mechanical — state inflation *is* part of national inflation. If we include national inflation as a control in a regression where state inflation is the dependent variable, the slope on national inflation will be near 1 and the labor-market coefficient will essentially be picking up only the *state-specific deviation* from the national trend. That's actually a defensible specification (it's effectively a "relative" Phillips Curve), but we should be aware of what we're estimating.
- **Fed funds rate ↔ unemployment rate: ~−0.54 in all three states.** Below the |0.7| red line, but worth watching. The Fed cuts rates when unemployment is rising, so this correlation is endogenous; we should be cautious interpreting either coefficient causally.

Other notable pairs — none above |0.7| within state — include fed funds × mortgage rate (positive but moderate, ~0.5 in raw checks) and HPI × national inflation (~0.4). The remaining correlations are all comfortably under the multicollinearity threshold.

**Bottom line.** Oil and national inflation are the two strongest correlates of state inflation in every state — both should be in the regression. National inflation is so highly correlated with state inflation that we need to think carefully about whether to include it (as a "deviation" specification) or absorb the national trend with time fixed effects instead. We'll resolve this in Week 3.

## Distribution of Key Variables

In [ ]:
STATE_COLORS = {
    'TX': '#2ca02c',  # green
    'MA': '#1f77b4',  # blue
    'OH': '#d62728',  # red
}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for col, (code, name) in enumerate(state_order):
    df = panel[panel['state'] == code]
    color = STATE_COLORS[code]

    ax_u = axes[0, col]
    u = df['unemployment_rate'].dropna()
    sns.histplot(u, bins=25, kde=True, ax=ax_u,
                 color=color, edgecolor='white', alpha=0.75)
    ax_u.axvline(u.mean(), color='black', linestyle='--', linewidth=1.5,
                 label=f'Mean = {u.mean():.2f}%')
    ax_u.set_title(f'{name} — Unemployment Rate', fontsize=13, fontweight='bold')
    ax_u.set_xlabel('Unemployment Rate (%)')
    ax_u.set_ylabel('Count' if col == 0 else '')
    ax_u.legend(loc='upper right', fontsize=10)

    ax_i = axes[1, col]
    i = df['inflation_rate_yoy'].dropna()
    sns.histplot(i, bins=25, kde=True, ax=ax_i,
                 color=color, edgecolor='white', alpha=0.75)
    ax_i.axvline(i.mean(), color='black', linestyle='--', linewidth=1.5,
                 label=f'Mean = {i.mean():.2f}%')
    ax_i.axvline(2, color='gray', linestyle=':', linewidth=1.5,
                 label='Fed target = 2%')
    ax_i.set_title(f'{name} — YoY Inflation Rate', fontsize=13, fontweight='bold')
    ax_i.set_xlabel('YoY Inflation Rate (%)')
    ax_i.set_ylabel('Count' if col == 0 else '')
    ax_i.legend(loc='upper right', fontsize=10)

fig.suptitle('Distributions of Unemployment and Inflation by State',
             fontsize=16, fontweight='bold', y=1.00)
fig.tight_layout()

fig.savefig('../figures/distributions.png', dpi=200, bbox_inches='tight')

plt.show()

### Distribution shape

**None of these distributions are normal — they are all positively skewed**, and the source of the skew is the same in every panel: large outlier events at the upper tail.

- **Unemployment is right-skewed in every state**, with skewness ≈ 1.2 for TX, 1.5 for OH, and an extreme 2.2 for MA driven by the COVID spike to 17.8%. The bulk of months sit between 3.5% and 6%, but recession-era observations pull a long right tail that any normality-assuming model will need to account for (heteroskedasticity, possibly robust standard errors in Week 3).
- **Inflation is also right-skewed**, especially in TX and OH, where the 2021–22 surge produces a visible right tail well above the Fed's 2% target. Massachusetts is closest to symmetric (skew = 0.53), again reflecting its more stable price dynamics. In all three states the mode sits a bit below 2%, but the **mean is pulled above 2%** by the COVID-era right tail — exactly what you'd expect after the largest inflation shock in 40 years.
- **State-shape comparison.** Massachusetts has the most extreme *unemployment* outlier (the COVID peak) but the most well-behaved *inflation* distribution. Texas is the inverse: its unemployment distribution is relatively well-behaved but its inflation has the heaviest right tail. Ohio sits between the two on both dimensions. The pattern mirrors what we saw in the time-series and Phillips Curve figures — MA's *labor market* is the volatile thing about MA, while TX's *prices* are the volatile thing about TX.

For the Week 3 regression, the practical implication is that we should use **heteroskedasticity-robust standard errors** rather than the default OLS errors, since the residuals will almost certainly inherit the right-skew of the dependent variable. Log-transformations of oil and HPI may also help symmetrize the regressors.

## Deep Dive: Oil Prices and the Texas Phillips Curve

Texas is the most interesting state in the panel for one reason: **oil**. The oil and gas sector is a large share of Texas employment, capital investment, and consumer prices in a way it simply isn't for Massachusetts or Ohio. This creates a specific mechanism that can *contaminate* the Phillips Curve relationship in TX in a very particular way.

When oil prices **rise sharply**, two things happen in Texas at the same time:

1. **Energy companies hire** — drilling, services, midstream — which *reduces* Texas unemployment.
2. **Energy costs flow into consumer prices** (gasoline, transport, plastics), which *increases* Texas inflation.

Notice the direction: low unemployment *and* high inflation move together. That looks superficially like the Phillips Curve (negative slope), but the cause is on the *supply* side — oil — not the labor market. A clean Phillips Curve mechanism would have lower unemployment causing wage and price pressure through tight labor; the oil mechanism gets there for an entirely different reason.

When oil prices **crash**, the same logic runs in reverse: Texas energy jobs disappear (unemployment rises) and energy-driven inflation falls. Once again, the two variables move together in a way that looks Phillips-Curve-shaped, but is really an oil story.

To disentangle this, we do two things below: (1) plot the three series together as z-scores so we can see when oil, unemployment, and inflation co-move; and (2) split the TX scatter into oil-price regimes (low / medium / high) and fit a Phillips Curve within each, to see whether the relationship is genuinely different at different points in the oil cycle.

In [ ]:
tx = panel[panel['state'] == 'TX'].sort_values('date').copy()

def zscore(s):
    return (s - s.mean()) / s.std()

tx['z_unemp']     = zscore(tx['unemployment_rate'])
tx['z_inflation'] = zscore(tx['inflation_rate_yoy'])
tx['z_oil']       = zscore(tx['oil_price_wti'])

q25, q75 = tx['oil_price_wti'].quantile([0.25, 0.75])

def oil_regime(x):
    if pd.isna(x):
        return np.nan
    if x < q25:
        return 'Low'
    if x > q75:
        return 'High'
    return 'Medium'

tx['oil_regime'] = tx['oil_price_wti'].apply(oil_regime)

REGIME_ORDER = ['Low', 'Medium', 'High']
REGIME_COLORS = {
    'Low':    '#1f77b4',  # blue
    'Medium': '#7f7f7f',  # gray
    'High':   '#d62728',  # red
}

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

ax1 = axes[0]
ax1.plot(tx['date'], tx['z_unemp'],
         color='#1f3a93', linewidth=2, label='Unemployment (z)')
ax1.plot(tx['date'], tx['z_inflation'],
         color='#a52a2a', linewidth=2, label='Inflation YoY (z)')
ax1.plot(tx['date'], tx['z_oil'],
         color='#b8860b', linewidth=2, label='WTI Oil Price (z)')
ax1.axhline(0, color='black', linewidth=0.8, alpha=0.5)
for start, end in recessions:
    ax1.axvspan(pd.Timestamp(start), pd.Timestamp(end),
                color='gray', alpha=0.2, zorder=0)
ax1.set_ylabel('Standard Deviations from Mean (Z-Score)')
ax1.set_xlabel('Date')
ax1.set_title('Texas: Unemployment, Inflation, and Oil Prices (Standardized)',
              fontsize=13, fontweight='bold')
ax1.legend(loc='upper left', ncol=3, frameon=True)
ax1.xaxis.set_major_locator(mdates.YearLocator(2))
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax2 = axes[1]
regime_results = {}
for regime in REGIME_ORDER:
    df_r = tx[(tx['oil_regime'] == regime)
              & tx['inflation_rate_yoy'].notna()
              & tx['unemployment_rate'].notna()]
    if len(df_r) == 0:
        continue
    color = REGIME_COLORS[regime]
    ax2.scatter(df_r['unemployment_rate'], df_r['inflation_rate_yoy'],
                color=color, alpha=0.55, s=45, edgecolors='none',
                label=f'{regime} oil (n={len(df_r)})')

    if len(df_r) >= 3:
        reg = stats.linregress(df_r['unemployment_rate'], df_r['inflation_rate_yoy'])
        x_line = np.linspace(df_r['unemployment_rate'].min(),
                             df_r['unemployment_rate'].max(), 50)
        ax2.plot(x_line, reg.intercept + reg.slope * x_line,
                 color=color, linewidth=2.5)
        regime_results[regime] = (reg.slope, reg.rvalue ** 2, reg.pvalue, len(df_r))

ax2.axhline(2, color='gray', linestyle='--', linewidth=1, alpha=0.6)
ax2.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.6)
ax2.set_xlabel('Unemployment Rate (%)')
ax2.set_ylabel('YoY Inflation Rate (%)')
ax2.set_title(f'Texas Phillips Curve by Oil Price Regime '
              f'(Low < \\${q25:.0f}/bbl; High > \\${q75:.0f}/bbl)',
              fontsize=13, fontweight='bold')
ax2.legend(loc='upper right', frameon=True)

fig.tight_layout()
fig.savefig('../figures/texas_oil_deep_dive.png', dpi=200, bbox_inches='tight')

plt.show()

print('\nOil-regime Phillips Curve slopes (Texas):')
for regime, (slope, r2, pval, n) in regime_results.items():
    print(f'  {regime:<6}  slope={slope:+.3f}  R^2={r2:.3f}  p={pval:.3g}  n={n}')

### What the Texas oil deep dive shows

**The z-score time series makes the oil mechanism visible.** Compare the three series during the boom-and-bust episodes:

- **2008 oil spike → 2009 crash.** Oil rises above +2σ in 2008, drags Texas inflation above +2σ shortly after, and once oil collapses in late 2008 both series fall together while unemployment climbs above +2σ. Oil and inflation co-move tightly; unemployment moves with a lag.
- **2014–2016 oil bust.** Oil falls below −1σ in 2014 and stays there for two years. Texas inflation drops to ~0σ. Texas unemployment, which had been below the mean during the prior boom, ticks back up. This is the classic Texas oil-bust signature, separate from the national business cycle.
- **2020–2022.** Oil swings from a brief crash to a major spike. Inflation tracks oil to the upside, and unemployment falls from its COVID peak. All three move together, which is exactly what a supply-driven episode is supposed to look like.

**The oil-regime scatter sharpens the picture quantitatively.** Splitting TX into three regimes by oil price (cutoffs at the 25th and 75th percentiles, roughly \$45 and \$82 per barrel) reveals that the apparent Phillips Curve slope in Texas depends heavily on which oil regime you're in:

- **Low-oil regime (slope ≈ −0.32, R² ≈ 0.12, p = 0.003).** When oil is cheap, the Phillips Curve is *present and significant* in Texas — roughly the same slope as the all-period TX estimate. The labor-market mechanism is visible because the oil mechanism is dormant.
- **Medium-oil regime (slope ≈ −0.17, R² ≈ 0.02, p = 0.09).** When oil is in a normal range, the relationship is **statistically insignificant and economically tiny**. The labor-market signal essentially disappears — most of these months are the 2010s Long Expansion, when low unemployment failed to produce much inflation.
- **High-oil regime (slope ≈ −0.90, R² ≈ 0.32, p ≪ 0.001).** When oil is expensive, the Phillips Curve looks **dramatically steeper** than the pooled estimate. But this is almost certainly the oil-driven co-movement: high oil ↔ inflation surge ↔ energy-job hiring ↔ low unemployment. The "Phillips Curve" we'd estimate from these months alone is largely a supply-side artifact.

**The implication for Week 3.** The Texas Phillips Curve slope is not a single number. It is a function of the oil environment, and the apparent steepness during high-oil months reflects a *supply* mechanism that has nothing to do with the labor-market story the textbook curve tries to capture. That is the empirical case for including **WTI oil price as an explicit control** in the panel regression: without it, the TX coefficient will absorb the oil channel and over-state the labor-market component of inflation. With oil controlled, we should see the TX coefficient move toward the cleaner labor-market value the low-oil regime suggests.

## Cross-State Comparison

We've looked at each state in isolation; the final figure of the notebook pulls the headline number — the Phillips Curve **slope** itself — into a single side-by-side comparison. The left panel shows the pooled slope for each state with a 95% confidence interval. The right panel shows how that slope has changed over time within each state, era by era. Together, these two charts encode the central empirical finding of Week 2.

In [ ]:
pooled_results = {}
for code, _ in state_order:
    d = panel[(panel['state'] == code)
              & panel['inflation_rate_yoy'].notna()
              & panel['unemployment_rate'].notna()]
    reg = stats.linregress(d['unemployment_rate'], d['inflation_rate_yoy'])
    pooled_results[code] = {
        'slope': reg.slope,
        'se': reg.stderr,
        'ci_low': reg.slope - 1.96 * reg.stderr,
        'ci_high': reg.slope + 1.96 * reg.stderr,
        'n': len(d),
    }

era_results = {}
for code, _ in state_order:
    for era in ERA_ORDER:
        d = panel[(panel['state'] == code)
                  & (panel['era_short'] == era)
                  & panel['inflation_rate_yoy'].notna()
                  & panel['unemployment_rate'].notna()]
        if len(d) < 3:
            continue
        reg = stats.linregress(d['unemployment_rate'], d['inflation_rate_yoy'])
        era_results[(code, era)] = {
            'slope': reg.slope,
            'se': reg.stderr,
        }

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax1 = axes[0]
codes = [c for c, _ in state_order]
names = [n for _, n in state_order]
slopes = [pooled_results[c]['slope'] for c in codes]
errs   = [1.96 * pooled_results[c]['se'] for c in codes]
colors = [STATE_COLORS[c] for c in codes]

bars = ax1.bar(names, slopes, yerr=errs, color=colors,
               edgecolor='black', linewidth=1, capsize=8,
               error_kw={'elinewidth': 1.5, 'ecolor': 'black'})
ax1.axhline(0, color='black', linestyle='--', linewidth=1)

for bar, slope in zip(bars, slopes):
    ax1.text(bar.get_x() + bar.get_width() / 2,
             slope - 0.025,
             f'{slope:+.3f}',
             ha='center', va='top', fontsize=11, fontweight='bold', color='white')

ax1.set_ylabel('Phillips Curve Slope ($\\beta_1$)')
ax1.set_title('Estimated Phillips Curve Slope by State\n(2000–Present, 95% CI)',
              fontsize=13, fontweight='bold')
ax1.set_ylim(min(slopes) - 0.2, max(0.05, max(slopes) + 0.1))

ax2 = axes[1]
x = np.arange(len(ERA_ORDER))
width = 0.25

for i, (code, name) in enumerate(state_order):
    era_slopes = [era_results.get((code, e), {}).get('slope', np.nan)
                  for e in ERA_ORDER]
    era_errs   = [1.96 * era_results.get((code, e), {}).get('se', 0)
                  for e in ERA_ORDER]
    offset = (i - 1) * width
    ax2.bar(x + offset, era_slopes, width,
            yerr=era_errs, color=STATE_COLORS[code],
            edgecolor='black', linewidth=0.8, capsize=4,
            label=name,
            error_kw={'elinewidth': 1.0, 'ecolor': 'black', 'alpha': 0.7})

ax2.axhline(0, color='black', linestyle='--', linewidth=1)
ax2.set_xticks(x)
ax2.set_xticklabels([e.replace(' & ', ' &\n') for e in ERA_ORDER], fontsize=10)
ax2.set_ylabel('Phillips Curve Slope ($\\beta_1$)')
ax2.set_title('Phillips Curve Slope Over Time by State',
              fontsize=13, fontweight='bold')
ax2.legend(loc='upper right', frameon=True, fontsize=10)

fig.tight_layout()
fig.savefig('../figures/slope_comparison.png', dpi=200, bbox_inches='tight')

plt.show()

print('\nPooled slopes:')
for c, _ in state_order:
    r = pooled_results[c]
    print(f'  {c}: {r["slope"]:+.3f}  [{r["ci_low"]:+.3f}, {r["ci_high"]:+.3f}]  n={r["n"]}')
print('\nEra-by-state slopes:')
for c, _ in state_order:
    for e in ERA_ORDER:
        r = era_results.get((c, e))
        if r is None:
            continue
        print(f'  {c} | {e:<20} slope={r["slope"]:+.3f}  se={r["se"]:.3f}')

### Headline finding: state-level Phillips Curves are similar in level but very different over time

**Pooled, the three states look almost identical.** Massachusetts (−0.346), Texas (−0.354), and Ohio (−0.358) all sit within a few hundredths of each other, and their 95% confidence intervals overlap heavily ([−0.44, −0.25], [−0.49, −0.21], and [−0.47, −0.25] respectively). On the *average* slope over 25 years, there is no statistically meaningful difference between the three states — every confidence interval clearly excludes zero, but none excludes the others. Looked at this way, the Phillips Curve is alive, modest in magnitude, and roughly the same in TX, MA, and OH.

**Once we break the sample by era, the apparent unity dissolves.** The era-by-state bar chart is the most important visualization in this notebook. Four observations:

1. **The Great Recession slope is uniformly steep and uniformly negative**, with magnitudes of roughly **−1.2 to −1.3** in every state. Recessions reveal the curve.
2. **The Long Expansion slope is positive — and statistically distinguishable from negative — in Texas (+0.25) and Ohio (+0.26), and essentially zero in Massachusetts (+0.02).** In none of the three states does the 2010s slope match the textbook prediction. This era is where the Phillips Curve "broke" at the state level, exactly mirroring the national debate.
3. **COVID & Aftermath restores a negative slope** in every state, but at very different magnitudes: **TX −0.72, OH −0.58, MA −0.39**. Texas's slope is nearly twice Massachusetts's, with confidence intervals that probably do not overlap — the first place in the analysis where we see a *real* cross-state difference.
4. **Pre-Crisis slopes are mildly negative across the board** (−0.24 to −0.43), consistent with the "weak but present" Phillips Curve characterization of the 2000–2007 period.

**Which state has the steepest curve?** It depends entirely on the era. Pooled: essentially a tie. By era: Massachusetts (in 2008–09), Ohio (in pre-crisis), or Texas (in COVID) take turns at the top.

**Which state has the flattest?** Massachusetts during the Long Expansion is the flattest single estimate in the entire panel — a slope of +0.017, indistinguishable from zero, the canonical "dead Phillips Curve" observation at state level.

**Setup for Week 3.** Two patterns will need to be tested formally:
- **Era × state interactions are real.** A pooled panel regression that ignores them will average the −1.3 of the Great Recession with the +0.25 of the Long Expansion and report a moderate negative number that obscures both. Week 3 must include either era dummies or a flexible time control.
- **TX's COVID slope is much steeper than MA's.** This is the empirical claim that motivates including oil and state characteristics in the model — if it holds up after controlling for WTI, it points to genuine cross-state heterogeneity in the curve. If it disappears once oil is controlled for, it confirms the supply-shock interpretation we developed in the TX deep dive.

These two findings — *the slope shifts dramatically over time*, and *the COVID-era slope differs across states in a way the rest of the sample doesn't* — are the central empirical claims that Week 3's regressions will be designed to test.

## Exploratory Analysis — Key Takeaways

Pulling the threads from all of the figures and statistics in this notebook together, the exploratory analysis points to six headline findings:

1. **The Phillips Curve is alive in the pooled data, but weak.** When we run a simple OLS regression of YoY inflation on unemployment over the full 2000–present window, all three states return a negative slope: **TX −0.35, MA −0.35, OH −0.36** — every estimate is highly statistically significant (p ≪ 0.001) and every 95% confidence interval excludes zero. But the R² values are only **0.08 (TX), 0.15 (MA), and 0.13 (OH)**, meaning unemployment alone explains less than one-sixth of monthly state inflation. The textbook relationship is present in the *sign* but lives inside a lot of noise that controls will need to absorb.

2. **Cross-state differences in the *level* of the curve are small; differences in *fit* are real.** Massachusetts has the strongest unemployment-inflation correlation (−0.39) and the cleanest single-variable Phillips Curve, plausibly because its service-heavy economy passes labor-market pressure into prices most directly. Texas has the weakest (−0.28), consistent with its CPI being dragged around by oil far more than by local labor markets. Ohio sits between the two. The 95% CIs on the pooled slopes overlap heavily, so on a single number the three states are statistically interchangeable; the difference shows up in how much variance the labor-market variable explains, not in how steep the slope is.

3. **The slope has shifted dramatically across economic eras — and is the central empirical story of this notebook.** Splitting the sample into four regimes reveals four very different Phillips Curves:
    - **Great Recession (2008–09)**: slopes of **−1.2 to −1.3** with R² 0.57–0.69 — textbook negative demand shock; the curve is at its steepest and best-fit here.
    - **Long Expansion (2010–19)**: slope ≈ **0 in MA (+0.02), positive in TX and OH (+0.25)**. This is the "is the Phillips Curve dead?" episode that drove the national debate of the late 2010s, and it replicates clearly at state level.
    - **COVID & Aftermath (2020+)**: slope returns to negative (**TX −0.72, OH −0.58, MA −0.39**) but the *level* of the entire scatter shifts upward — inflation 6–10% at unemployment rates similar to the previous era's 2%.
    - **Pre-Crisis (2000–07)**: mild negative (−0.24 to −0.43), low R², a baseline against which the later regimes can be compared.

4. **The Texas Phillips Curve is contaminated by oil.** Splitting TX by oil-price regime (low/medium/high, at the 25th and 75th percentiles, ≈ \$45 and \$82 per barrel) yields three very different slopes: **−0.32 when oil is low, −0.17 (insignificant) when oil is mid-range, −0.90 when oil is high.** The "steepest" Texas Phillips Curve appears precisely when oil is at its most disruptive — i.e., when oil itself is moving inflation and unemployment in the same direction, mimicking the curve through a supply-side mechanism. Oil correlates 0.43 with TX inflation (the highest of any control variable in the panel, tied with OH), confirming that WTI must appear as an explicit regressor in Week 3 if we want to isolate the *labor-market* component of TX's curve.

5. **The COVID inflation surge does not look like the Phillips Curve at work — it looks like a level shift.** In the era-colored scatter, the 2020+ orange cluster sits at *the same unemployment levels* as the 2010s green cluster (3–6%) but at **much higher inflation (6–10% vs ~2%)**. A stable Phillips Curve would have predicted inflation near 2% at those unemployment rates; instead, the entire relationship moved up the page. This is the visual signature of a supply-side shock (oil, supply chains, goods shortages) rather than a labor-market shock — and it is the *primary* reason the simple pooled regression is the wrong specification.

6. **The empirical findings dictate the structure of the Week 3 regression.** The exploratory results point to a specific model rather than a generic "panel regression":
    - **State fixed effects**, to absorb the level differences in average inflation and unemployment across MA, OH, and TX visible in the summary statistics.
    - **Explicit controls for WTI oil, the federal funds rate, the 30-year mortgage rate, and national inflation**, since these account for the bulk of the variation that the single-variable Phillips Curve misses — particularly the supply-side shocks driving the COVID era.
    - **Era dummies or a time control** to handle the dramatic slope shifts across the four regimes; pooling them together averages a −1.3 with a +0.25 and yields a slope that describes none of the actual episodes.
    - **Era × state (or oil × state) interactions** to formally test the heterogeneous COVID slope (TX −0.72 vs MA −0.39) and to test whether TX's apparent steepness in high-oil months survives once oil is held constant.
    - **Heteroskedasticity-robust standard errors**, given the right-skewed residuals visible in the distribution figures.

## Next Steps

This concludes the exploratory analysis. The next notebook — `03_regression_analysis.ipynb` — will move from visualization to formal estimation. Specifically, it will:

- **Estimate simple OLS Phillips Curves by state**, in a regression framework with proper standard errors, replicating the bivariate slopes shown in this notebook as a sanity check and a baseline.
- **Add multiple controls** — WTI oil price, the 30-year mortgage rate, national CPI inflation, and the Home Price Index — to isolate the *labor-market* component of state inflation from the supply-side and macroeconomic forces this notebook surfaced as dominant.
- **Move to a panel specification with state fixed effects**, pooling MA, OH, and TX while absorbing the level differences in average inflation and unemployment across the three states.
- **Run sub-period (era) regressions and era × labor-market interactions** to formally test whether the slope changes documented above — from −1.3 in the Great Recession to ~0 in the Long Expansion to a renewed negative number during COVID — are statistically distinguishable from each other.
- **Run an oil × state interaction for Texas** to formally test whether the apparent steepness of the TX Phillips Curve in high-oil months reflects a genuine labor-market relationship or, as the deep dive suggests, an oil-driven supply-side artifact.
- **Use heteroskedasticity-robust standard errors throughout**, given the right-skewed distributions of both unemployment and inflation.

The result will be a set of regression tables that formally test the empirical claims this notebook has *visualized* — and a quantitative answer to the question this project set out to ask: is the Phillips Curve still alive at the U.S. state level, and if so, where, when, and under what conditions?